# A small MLP, three ways: CUDA Fortran vs Rust vs CuPy

`tensor_core_engine_v5.2` is benchmarked op by op in
`benchmark_v5_2_*_fortran_vs_rust.ipynb`. This notebook is the other half of the
question: on a **real model and a real dataset**, does any of it matter?

The workload is batch inference for a 3-layer MLP trained on `sklearn`'s
handwritten-digit set. Each fully connected layer is exactly one fused
`GEMM + bias + ReLU`, which is the operation `batched_matmul_bias_relu` exists
for, so the model maps onto the engine without being contorted to fit it.

Three backends compute the identical forward pass:

| backend | how a layer is computed |
|---|---|
| `cupy (FP64)` | `cp.matmul` + separate bias/ReLU kernels, FP64 throughout |
| `cupy (FP32)` | the same, but narrowed to FP32 for the GEMM and widened back |
| `fortran` | `cuda_matlib.so` — one fused cuBLASLt epilogue |
| `rust` | `librust_matlib.so` — the same, via `rustc_codegen_cuda` |
| `fortran (FP32)`, `rust (FP32)` | the v5.3 FP32-in/FP32-out entry points |
| `fortran (FP32, F-order)`, `rust (FP32, F-order)` | the same, with activations kept column-major |

**Both CuPy baselines matter.** The engines compute the GEMM in FP32, and this
is a consumer GPU where FP64 runs at 1/64 rate — so comparing them against FP64
CuPy mostly measures the precision tier, not the engine. `cupy (FP32)` does the
same narrowing the engines do internally, so *that* is the like-for-like
baseline, and the gap to it is what fusion and dispatch are actually worth.

Two things are worth watching:

1. **Accuracy** — the engines compute the GEMM in FP32, not FP64. Since v5.2
   (D10) that is *exact* FP32 rather than TF32, so the error is ~1e-7 instead
   of ~1e-4. The last section asks the only question a practitioner actually
   cares about: **does the tier ever change a prediction?**
2. **Throughput** — the fused path saves a global-memory round trip per layer,
   but the FP64 entry points make a *chained* model pay a narrow-and-widen on
   every layer. Which effect wins is not obvious from the op-level benchmark,
   and it is the more interesting half of this notebook. The answer here is not
   the flattering one — and the FP32 entry points, which this notebook is the
   reason v5.3 has, only close part of the gap.

In [1]:
# Two environment variables must be set BEFORE cuda_matlib.so / cupy load.
import os
os.environ.setdefault("NV_ACC_MEM_MANAGE", "0")   # nvfortran's device pool hides frees

import sys
sys.path.insert(0, os.getcwd())
from cublas_preload import preload_system_cublas
preload_system_cublas()                            # system cuBLAS before CuPy's bundled one

import numpy as np
import cupy as cp
import time
import matplotlib.pyplot as plt

from tensor_matrix_ops import TensorMatrixOps
from rust_matrix_ops import RustMatrixOps

fortran = TensorMatrixOps()
rust = RustMatrixOps()
print(f"rust_matlib version {rust.version}")

# Mobile/desktop GPUs start at a low boost clock; time nothing until it has ramped.
_w = cp.random.random((1024, 1024), dtype=cp.float64)
for _ in range(20):
    _ = _w @ _w
cp.cuda.Stream.null.synchronize()
del _w, _
cp.get_default_memory_pool().free_all_blocks()
print("GPU warm")

preloaded system cuBLAS from /usr/local/cuda-13.4/lib64


Initializing CUDA...
CUDA initialization complete
Loading library: /var/home/fraser/machine_learning/fortran/examples/collected_examples/matrix_dot/tensor13/tensor_core_engine_v5/cuda_matlib.so
Function signatures configured
Initializing CUDA resources (one-time operation)...
CUDA resources initialized
rust_matlib version 4


GPU warm


## The data

`load_digits` is 8x8 grayscale digits — small, real, and bundled with sklearn, so
this notebook needs no network. Features are standardised; the split is
stratified so the test set has every class.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_all, y_all = digits.data.astype(np.float64), digits.target.astype(np.int64)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.25, random_state=0, stratify=y_all)

mu, sigma = X_tr.mean(axis=0), X_tr.std(axis=0)
sigma[sigma == 0] = 1.0
X_tr = (X_tr - mu) / sigma
X_te = (X_te - mu) / sigma

# Columns are samples throughout: a layer is W @ X + b, which is the
# (m=out, k=in, n=batch) shape the engines' fused epilogue takes.
Xtr_g = cp.asarray(np.ascontiguousarray(X_tr.T))
Xte_g = cp.asarray(np.ascontiguousarray(X_te.T))
ytr_g = cp.asarray(y_tr)
yte_g = cp.asarray(y_te)

N_IN, N_CLASS = X_tr.shape[1], 10
print(f"train {X_tr.shape[0]} samples, test {X_te.shape[0]}, {N_IN} features, {N_CLASS} classes")
print(f"GPU layout  Xtr {Xtr_g.shape} (features x samples)")

train 1347 samples, test 450, 64 features, 10 classes
GPU layout  Xtr (64, 1347) (features x samples)


## Train the reference model

Plain FP64 CuPy, so the weights owe nothing to any engine — they are the common
input all three backends are then asked to reproduce. The network is
deliberately **wide** (64 -> 1024 -> 1024 -> 10): a narrow one would be too small
for the GPU to be the interesting part, and the point here is the GEMM.

In [3]:
rng = np.random.default_rng(0)
H = 1024
dims = [(H, N_IN), (H, H), (N_CLASS, H)]

def he_init(shape):
    fan_in = shape[1]
    return cp.asarray(rng.standard_normal(shape) * np.sqrt(2.0 / fan_in))

W = [he_init(s) for s in dims]
B = [cp.zeros((s[0], 1)) for s in dims]

def forward_train(X):
    """Reference FP64 forward pass; keeps activations for the backward pass."""
    a0 = X
    z1 = W[0] @ a0 + B[0]; a1 = cp.maximum(z1, 0.0)
    z2 = W[1] @ a1 + B[1]; a2 = cp.maximum(z2, 0.0)
    z3 = W[2] @ a2 + B[2]
    return a0, z1, a1, z2, a2, z3

def softmax_xent(logits, y):
    shifted = logits - logits.max(axis=0, keepdims=True)
    expz = cp.exp(shifted)
    p = expz / expz.sum(axis=0, keepdims=True)
    loss = float(-cp.mean(cp.log(p[y, cp.arange(y.size)] + 1e-12)))
    return p, loss

onehot = cp.zeros((N_CLASS, ytr_g.size))
onehot[ytr_g, cp.arange(ytr_g.size)] = 1.0

lr, epochs = 0.05, 300
for ep in range(epochs):
    a0, z1, a1, z2, a2, z3 = forward_train(Xtr_g)
    p, loss = softmax_xent(z3, ytr_g)
    m = ytr_g.size
    d3 = (p - onehot) / m
    d2 = (W[2].T @ d3) * (z2 > 0)
    d1 = (W[1].T @ d2) * (z1 > 0)
    for Wi, Bi, di, ai in ((W[2], B[2], d3, a2), (W[1], B[1], d2, a1), (W[0], B[0], d1, a0)):
        Wi -= lr * (di @ ai.T)
        Bi -= lr * di.sum(axis=1, keepdims=True)
    if ep % 60 == 0 or ep == epochs - 1:
        acc = float(cp.mean(cp.argmax(z3, axis=0) == ytr_g))
        print(f"epoch {ep:4d}  loss {loss:.4f}  train acc {acc:.4f}")

_, _, _, _, _, logits_te = forward_train(Xte_g)
ref_logits = logits_te
ref_pred = cp.argmax(ref_logits, axis=0)
print(f"\nFP64 reference test accuracy: {float(cp.mean(ref_pred == yte_g)):.4f}")

epoch    0  loss 2.8389  train acc 0.1507


epoch   60  loss 0.0697  train acc 0.9918


epoch  120  loss 0.0345  train acc 0.9978


epoch  180  loss 0.0221  train acc 0.9993


epoch  240  loss 0.0160  train acc 1.0000


epoch  299  loss 0.0125  train acc 1.0000

FP64 reference test accuracy: 0.9800


## The three backends

Identical arithmetic, three implementations. The engines take the weight matrix
and the activation batch as `(1, out, in)` and `(1, in, batch)` and return
`relu(W @ X + b)` from a single fused kernel; the last layer has no activation,
so it uses `batched_matmul_bias` instead.

In [4]:
def infer_cupy(X):
    """FP64 reference: GEMM, then a separate bias+ReLU kernel per layer."""
    a = X
    a = cp.maximum(W[0] @ a + B[0], 0.0)
    a = cp.maximum(W[1] @ a + B[1], 0.0)
    return W[2] @ a + B[2]

def _engine_infer(ops, X):
    """One fused GEMM+bias+ReLU per hidden layer, fused GEMM+bias for the output."""
    a = X
    for i in (0, 1):
        a = ops.batched_matmul_bias_relu(W[i][None], a[None], B[i].ravel())[0]
    return ops.batched_matmul_bias(W[2][None], a[None], B[2].ravel())[0]

def infer_fortran(X):
    return _engine_infer(fortran, X)

def infer_rust(X):
    return _engine_infer(rust, X)

def infer_cupy32(X):
    """Like-for-like baseline: same FP32 GEMM tier as the engines, but unfused.

    Takes and returns FP64 exactly as the engines do, narrowing internally,
    so the only differences left are the fused epilogue and the dispatch path.
    """
    a = X.astype(cp.float32)
    for i in (0, 1):
        a = cp.maximum(W[i].astype(cp.float32) @ a + B[i].astype(cp.float32), 0.0)
    return (W[2].astype(cp.float32) @ a + B[2].astype(cp.float32)).astype(cp.float64)

# v5.3: float32 in -> float32 out, dispatched on dtype by the same method.
# A deployed model would hold FP32 weights, so they are cast once, not per call.
W32 = [w.astype(cp.float32) for w in W]
B32 = [b.astype(cp.float32) for b in B]

def _engine_infer32(ops, X32):
    a = X32
    for i in (0, 1):
        a = ops.batched_matmul_bias_relu(W32[i][None], a[None], B32[i].ravel())[0]
    return ops.batched_matmul_bias(W32[2][None], a[None], B32[2].ravel())[0]

def infer_fortran32(X):
    return _engine_infer32(fortran, X.astype(cp.float32)).astype(cp.float64)

def infer_rust32(X):
    return _engine_infer32(rust, X.astype(cp.float32)).astype(cp.float64)

def _engine_infer32_forder(ops, X):
    a = cp.asfortranarray(X.astype(cp.float32))
    for i in (0, 1):
        a = ops.batched_matmul_bias_relu(W32[i], a, B32[i].ravel())
    return ops.batched_matmul_bias(W32[2], a, B32[2].ravel()).astype(cp.float64)

def infer_fortran32_forder(X):
    return _engine_infer32_forder(fortran, X)

def infer_rust32_forder(X):
    """As above, but the activation block stays column-major end to end.

    cuBLASLt's bias epilogue only emits column-major, so a C-order caller pays
    a transpose on the way out of every layer. Hand the activations in as
    F-order and that pass disappears: each layer's F-order result is already a
    valid F-order operand for the next. Weights stay C-order.
    """
    return _engine_infer32_forder(rust, X)

BACKENDS = {"cupy (FP64)": infer_cupy, "cupy (FP32)": infer_cupy32,
            "fortran": infer_fortran, "rust": infer_rust,
            "fortran (FP32)": infer_fortran32, "rust (FP32)": infer_rust32,
            "fortran (FP32, F)": infer_fortran32_forder,
            "rust (FP32, F)": infer_rust32_forder}

# Sanity: every backend must reproduce the reference logits and predictions.
print(f"{'backend':>14} {'max |logit diff|':>17} {'rel err':>10} {'pred flips':>11} {'test acc':>9}")
mag = float(cp.max(cp.abs(ref_logits)))
for name, fn in BACKENDS.items():
    out = fn(Xte_g)
    diff = float(cp.max(cp.abs(out - ref_logits)))
    pred = cp.argmax(out, axis=0)
    flips = int(cp.sum(pred != ref_pred))
    acc = float(cp.mean(pred == yte_g))
    print(f"{name:>14} {diff:17.3e} {diff / mag:10.2e} {flips:11d} {acc:9.4f}")

       backend  max |logit diff|    rel err  pred flips  test acc
   cupy (FP64)         0.000e+00   0.00e+00           0    0.9800
   cupy (FP32)         1.054e-05   4.31e-07           0    0.9800
       fortran         1.034e-05   4.22e-07           0    0.9800
          rust         1.034e-05   4.22e-07           0    0.9800
fortran (FP32)         1.034e-05   4.22e-07           0    0.9800
   rust (FP32)         1.034e-05   4.22e-07           0    0.9800
fortran (FP32, F)         1.034e-05   4.22e-07           0    0.9800
rust (FP32, F)         1.034e-05   4.22e-07           0    0.9800


## Throughput

Batch inference across a range of batch sizes. `min` of several timed runs, with
a full device sync on both sides of the timer — the same protocol the op-level
benchmark uses.

Read the two CuPy rows separately. **FP64 is the accuracy reference, not a
throughput peer** — this GPU runs FP64 at 1/64 rate, so most of the gap to it is
the precision tier the engines chose, not anything the engines do well.
**FP32 is the honest baseline**: same tier, same FP64 in/out, unfused. The
engine-vs-FP32 column is what the fused epilogue and the dispatch path are
worth.

The small batches are dominated by launch overhead (three kernel launches per
forward pass, a few microseconds each), so they say more about dispatch cost
than about the GPU.

In [5]:
def timed(fn, X, repeat=20):
    for _ in range(3):                      # warm the plan cache / workspace pool
        fn(X)
    cp.cuda.runtime.deviceSynchronize()
    ts = []
    for _ in range(repeat):
        cp.cuda.runtime.deviceSynchronize()
        t0 = time.perf_counter()
        fn(X)
        cp.cuda.Stream.null.synchronize()
        ts.append(time.perf_counter() - t0)
    return min(ts)

batch_sizes = [64, 256, 1024, 4096, 16384]
flops_per_sample = 2 * (H * N_IN + H * H + N_CLASS * H)

results = {name: [] for name in BACKENDS}
base = cp.asarray(np.ascontiguousarray(X_te.T))
print(f"{'batch':>7} " + " ".join(f"{n:>14}" for n in BACKENDS))
for nb in batch_sizes:
    reps = int(np.ceil(nb / base.shape[1]))
    Xb = cp.ascontiguousarray(cp.tile(base, (1, reps))[:, :nb])
    line = f"{nb:>7} "
    for name, fn in BACKENDS.items():
        t = timed(fn, Xb)
        gflops = flops_per_sample * nb / (t * 1e9)
        results[name].append((t, gflops, nb / t))
        line += f"{gflops:12.0f} GF "
    print(line)
    del Xb
    cp.get_default_memory_pool().free_all_blocks()

  batch    cupy (FP64)    cupy (FP32)        fortran           rust fortran (FP32)    rust (FP32) fortran (FP32, F) rust (FP32, F)
     64          162 GF          581 GF          462 GF          638 GF          417 GF          659 GF          862 GF          864 GF 


    256          200 GF         2229 GF         1406 GF         1880 GF         1285 GF         1933 GF         2461 GF         2482 GF 


   1024          220 GF         5442 GF         2513 GF         3460 GF         2495 GF         4126 GF         4307 GF         4338 GF 


   4096          219 GF         5684 GF         2744 GF         3623 GF         2875 GF         4450 GF         5267 GF         5252 GF 


  16384          220 GF         5715 GF         2902 GF         3504 GF         2955 GF         4174 GF         5573 GF         5498 GF 


In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(batch_sizes))
width = 0.105
for i, (name, rows) in enumerate(results.items()):
    ax1.bar(x + (i - 3.5) * width, [r[1] for r in rows], width, label=name)
ax1.set_xticks(x); ax1.set_xticklabels([str(b) for b in batch_sizes])
ax1.set_xlabel("batch size"); ax1.set_ylabel("GFLOPS")
ax1.set_title("MLP inference throughput (64-1024-1024-10)")
ax1.legend(); ax1.grid(True, axis="y", alpha=0.3)

base_gf = np.array([r[1] for r in results["cupy (FP32)"]])
for name in ("rust", "rust (FP32)", "fortran (FP32, F)", "rust (FP32, F)"):
    ax2.plot(x, np.array([r[1] for r in results[name]]) / base_gf, "o-", label=name)
ax2.axhline(1.0, color="0.4", lw=1, ls="--")
ax2.set_xticks(x); ax2.set_xticklabels([str(b) for b in batch_sizes])
ax2.set_xlabel("batch size"); ax2.set_ylabel("speedup vs CuPy FP32 (same tier)")
ax2.set_title("What fusion + dispatch are worth, tier held equal")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_mlp_backends.png", dpi=150, bbox_inches="tight")
plt.show()

/tmp/ipykernel_159735/473238750.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Where the time actually goes

The throughput table above is the opposite of what the op-level benchmark would
lead you to expect, so it is worth taking one layer apart. All of these compute
the same `relu(W @ X + b)` at the same shape.

In [7]:
m = k = H
n_big = 16384
W64 = cp.ascontiguousarray(W[1]); X64 = cp.random.random((k, n_big), dtype=cp.float64)
b64 = cp.ascontiguousarray(B[1].ravel())
W32b, X32b, b32 = W64.astype(cp.float32), X64.astype(cp.float32), b64.astype(cp.float32)
X32f = cp.asfortranarray(X32b)
fl = 2 * m * k * n_big

def ms(fn, reps=20):
    for _ in range(3):
        fn()
    cp.cuda.runtime.deviceSynchronize()
    ts = []
    for _ in range(reps):
        cp.cuda.runtime.deviceSynchronize()
        t0 = time.perf_counter(); fn(); cp.cuda.Stream.null.synchronize()
        ts.append(time.perf_counter() - t0)
    return min(ts) * 1e3

rows = [
    ("engine matmul, TF32 tier (FP64 in/out, no epilogue)", lambda: rust.matmul(W64, X64)),
    ("cupy FP32 GEMM alone (no casts, no bias/relu)", lambda: W32b @ X32b),
    ("cupy FP32 GEMM + bias + relu (no casts)", lambda: cp.maximum(W32b @ X32b + b32[:, None], 0.0)),
    ("cupy FP32 incl. casting both operands", lambda: cp.maximum(W64.astype(cp.float32) @ X64.astype(cp.float32) + b32[:, None], 0.0)),
    ("engine fused epilogue, exact FP32 (FP64 in/out)", lambda: rust.batched_matmul_bias_relu(W64[None], X64[None], b64)),
    ("engine fused epilogue, FP32 in/out (v5.3)", lambda: rust.batched_matmul_bias_relu(W32b[None], X32b[None], b32)),
    ("engine fused epilogue, FP32 + F-order activations", lambda: rust.batched_matmul_bias_relu(W32b, X32f, b32)),
    ("fortran fused, FP32 + F-order activations", lambda: fortran.batched_matmul_bias_relu(W32b, X32f, b32)),
]
print(f"one layer, ({m} x {k}) @ ({k} x {n_big})")
print(f"{'':<52} {'ms':>8} {'GFLOPS':>9}")
for label, fn in rows:
    t_ms = ms(fn)
    print(f"{label:<52} {t_ms:8.3f} {fl / (t_ms * 1e6):9.0f}")
print()
print(f"{'f64->f32 cast of X alone':<52} {ms(lambda: X64.astype(cp.float32)):8.3f}")
# What the FP32 path still cannot avoid: cuBLASLt's bias epilogue only accepts
# column-major layouts, so D always lands transposed relative to what the caller
# wants back. This is a proxy for that kernel's work.
Dcm = cp.asfortranarray(cp.random.random((m, n_big), dtype=cp.float32))
print(f"{'forced col-major -> row-major output transpose':<52} {ms(lambda: cp.ascontiguousarray(Dcm)):8.3f}")
del X64, X32b, X32f, Dcm
cp.get_default_memory_pool().free_all_blocks()

one layer, (1024 x 1024) @ (1024 x 16384)
                                                           ms    GFLOPS
engine matmul, TF32 tier (FP64 in/out, no epilogue)     2.866     11987
cupy FP32 GEMM alone (no casts, no bias/relu)           3.636      9449


cupy FP32 GEMM + bias + relu (no casts)                 4.978      6902
cupy FP32 incl. casting both operands                   5.674      6056


engine fused epilogue, exact FP32 (FP64 in/out)         6.864      5006
engine fused epilogue, FP32 in/out (v5.3)               5.983      5743


engine fused epilogue, FP32 + F-order activations       4.932      6966
fortran fused, FP32 + F-order activations               5.228      6572

f64->f32 cast of X alone                                0.805
forced col-major -> row-major output transpose          1.259


Three costs the op-level benchmark never shows up:

- **The FP64 boundary was paid per layer** — a narrow of both operands, the
  FP16 range guard's max-abs reduction, and a widen of the result, on every
  call. That is what the v5.3 FP32 entry point removes, and it is worth ~15% on
  Rust and ~5% on Fortran here.
- **The output transpose is removed by layout, not by precision.** cuBLASLt's
  bias epilogue only emits column-major, so a C-order caller pays a transpose
  on the way out of every layer — about as much as the bias+ReLU pass the
  fusion saves. Keeping the activations F-order deletes that pass outright:
  compare the last two rows, and note the saving matches the measured transpose
  cost almost exactly.
- **Exact FP32 is not free.** The engine's own TF32 `matmul` is the fastest row
  in the table — faster than its exact-FP32 fused epilogue at the same shape.
  That is the D10 trade seen from the other side: v5.2 bought ~1000x accuracy
  on this path and gave up throughput for it.

## Does the precision tier ever change a prediction?

This is the question the op-level accuracy columns cannot answer.

The engines offer the same GEMM at three tiers, and v5.2 pinned the fused
epilogue to the most accurate of them. Running the *same trained weights*
through each tier shows what the difference is worth in the only currency that
matters for a classifier: how many test digits change label.

Note `matmul`'s TF32 tier is what the epilogue used to be before v5.2 D10 — so
the last row is, in effect, the bug this engine shipped with.

In [8]:
def infer_tier(ops, X, tier):
    """Same forward pass, but the GEMM tier is chosen explicitly."""
    gemm = {"tc_split": ops.matmul_tc_split, "tf32": ops.matmul}[tier]
    a = X
    for i in (0, 1):
        a = cp.maximum(gemm(W[i], a) + B[i], 0.0)
    return gemm(W[2], a) + B[2]

rows = []
out = infer_rust(Xte_g)
rows.append(("fused epilogue (v5.2 default, exact FP32)", out))
rows.append(("matmul_tc_split (1 exact FP32 + 2 TF32)", infer_tier(rust, Xte_g, "tc_split")))
rows.append(("matmul TF32 (what the epilogue was pre-D10)", infer_tier(rust, Xte_g, "tf32")))

ref_acc = float(cp.mean(ref_pred == yte_g))
print(f"FP64 reference: test accuracy {ref_acc:.4f} on {yte_g.size} digits\n")
print(f"{'tier':>44} {'rel logit err':>14} {'pred flips':>11} {'test acc':>9}")
for name, o in rows:
    d = float(cp.max(cp.abs(o - ref_logits))) / mag
    pred = cp.argmax(o, axis=0)
    print(f"{name:>44} {d:14.2e} {int(cp.sum(pred != ref_pred)):11d} "
          f"{float(cp.mean(pred == yte_g)):9.4f}")

FP64 reference: test accuracy 0.9800 on 450 digits

                                        tier  rel logit err  pred flips  test acc
   fused epilogue (v5.2 default, exact FP32)       4.22e-07           0    0.9800
     matmul_tc_split (1 exact FP32 + 2 TF32)       3.04e-07           0    0.9800
 matmul TF32 (what the epilogue was pre-D10)       6.69e-04           0    0.9800


## What this shows

- **All four backends classify identically.** Every engine path reproduces the
  FP64 logits to ~4e-7 relative, and **zero** of the test digits change label —
  including the TF32 tier at ~7e-4. For this model FP64 was never what made it
  correct.
- **Two API-boundary costs, both now removable, and together they were the
  whole story.** The FP64 conversions went first (v5.3's FP32 entry points);
  the forced column-major output transpose went second, by keeping activations
  F-order. Each was worth roughly what the fused epilogue saves, which is why
  the engine started out *behind* unfused CuPy despite doing strictly less
  work. With both gone the fused path is level with CuPy FP32 at moderate
  batches and ahead of it at the largest ones. The op-level benchmark sees
  neither cost, because it measures one call at a time with both folded into
  that call.
- **Comparing against FP64 CuPy would have hidden it.** FP64 runs at 1/64 rate
  on this GPU, so the engines look 13-16x faster against that baseline and
  1.6-0.6x against the like-for-like FP32 one. Pick the baseline that matches
  the precision tier, or the number means nothing.

### What this example changed, and what is left

The FP32-in/FP32-out entry points (`rs_batched_matmul_bias_f32`,
`py_batched_matmul_bias_relu_f32`) exist because of this notebook. They are
dispatched automatically: pass float32 arrays to `batched_matmul_bias_relu` and
the conversion-free path is used.

The transpose went next, and it needed no new kernel — only permission to keep
the data where cuBLASLt already puts it. Pass an **F-order** 2-D activation
block and `batched_matmul_bias_relu` describes B column-major (TRANSB=N), writes
the epilogue straight into an F-order result, and skips the transpose entirely;
each layer's output is already a valid operand for the next. Weights stay
C-order.

Still open:

- **Chain several layers in one call**, so intermediate activations never return
  to the caller at all.

Fortran has the F-order path too, via a transposed-A variant in
`cublaslt_bridge.c` (`lt_matmul_bias_*_fp32_t`): A is consumed via OP_T, B is
already column-major, D lands column-major. That removes *both* its costs at
once — the row-major-to-column-major staging of A and B, which the C-order path
pays regardless of dtype, and the output transpose. It is the larger jump of the
two engines for that reason.

Where the engines *do* win today is what the op-level notebook says: single
large ops called from FP64 data, which is the MPDOK/GBLUP shape rather than the
deep-network shape.

Where the engines *do* win today is exactly where the op-level notebook says:
single large ops called from FP64 data, which is the MPDOK/GBLUP shape rather
than the deep-network shape.

One caveat on the accuracy conclusions: this network is wide relative to the
dataset, chosen so the GEMMs are large enough for the GPU to be the interesting
part, and its logits are well separated. A model whose decisions sit closer
together has more room for a low-precision tier to flip one — which is why the
tier is now a pinned choice rather than whatever the cuBLAS heuristic picked
(`V52_CHANGES.md`, D10).